# EDA for carbreakdown data

In deze notebook gaan we de dataset in in train_CarBreakDown onderzoeken. Eerst starten we met een algemeen zicht op de data set via een describe. Dit world gevolgd door null waarde weg te halen want deze zullen de eind resultaten het meest beïnvloeden. Dan kijken we de verdelingen van de data per kollom om te zien als er dingen opvallen. Als laatst kijken we nog voor enige andere eigenaardigheden.

Eerst starten we met het inlezen van de data. Hierop voeren we direct een describe uit om een snel overzicht van de data te verkrijgen.

In [ ]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# read csv file
df = pd.read_csv('../../csv/train/train_CarBreakDown.csv')
print(f"Null value overview:\n{df.isnull().sum()}\n")
print(f"Total nulls in dataset: {df.isnull().sum().sum()}")
print(f"Gemiddelde nulls in dataset: {round(df.isnull().sum().mean())}")

totaal_records = len(df)
print(f"Totaal aantal records {totaal_records}")
df.describe(include='all')

# initial suspicions:
# all collumns are missing values except id (logical) and breakdown _next_30_days which is our goal to calculate
# There negative values in mileage_km, engine_hours and cleanliness_score which should not be possible
# oil_quality is a percentage but max is 102...
# same with cleanliness_score
# maybe driver_satisfaction to

Uit de bovenstaande analyse is te zien dat 239 waarden over alle kollomen (1050) niet ingevuld is. Dit betekent dat we maximum 23% van de data verliezen, er van uit gaand dat elke null in een verschillend record voorkomt.

We kunnen hier op twee manier mee omgaan, amputatie of imputatie.

**Amputatie**: Het verwijderen van de rij of kollomen die null waarde bevatten om enkel over te blijven met ingevulde records. Dit is ten koste van de grote van de data set.

**Imputatie**: Het vervangen van de data, vaak door het gemiddelde of median om de data volledig te maken. Dit kan enkel als de rest van de gegevens ok zijn en kan de gegevens veralgemene.

 Zelfs als het niet exact 23%, 15% (893 resterend in dat geval) zou ook all veel impact hebben op de resultaten omdat de data set redelijk klein is en het houd ook niet rekening dat er voor nog andere redenen records gedropt kunnen worden. Het is dus het beste om zoveel mogelijk kollomen te imputeren. Dit gaat niet alteid handig zijn zoals bij vehicle brand, hier zal dan amputatie gebruikt worden.

## Per kollom null waarde behandeling

Nu gaan we per kollom af om te kijken wat er aangepast kan worden zoals het droppen/imputeren van null waarde. Er zijn twee soorten kollom in de dataset numerieke en categorische. 

Bij numerieke kollomen wordt er gekeken naar de verdeling van de waarde via een boxplot om te zien als er onwaarscheinlijke uitschieters zijn.

Bij categorische waarde kan dit natuurlijk niet en wordt er via een barchart gekeken hoe de verdeling is tussen categorieën. All deze kollomen hebben een "other" kollom en afhankelijk van de grote hier van kunnen de records hiervan ook gedropt worden omdat deze data niet specifiek is (het is well beter dan null). Dit maakt het bewaren van kollom waar het mogelijk is nog beter want dan kunnen other kollomen makkelijker gedropt worden.

### vehicle_brand

In [ ]:
vehicle_brand = df['vehicle_brand']
print(f"Kollom null waardes: {vehicle_brand.isnull().sum()} ({(vehicle_brand.isnull().sum()/totaal_records * 100).round(2)}%)")

print(df.vehicle_brand.value_counts())
df['vehicle_brand'].value_counts().plot(kind='bar',ylabel='Aantal voorkomen')


df.dropna(subset='vehicle_brand', inplace=True) # subset is gebruikt om enkel 1 kollom de records te droppen
df


Vehicle_brand is een **categorische kollom**, hierdoor wordt er zeker **geamputeerd**. Het heeft een boven gemiddeld aantal null waarde maar de gemiddelde category nemen zou niet een goede oplossing zijn want dan wordt deze over gerepresenteerd. De **other waarde blijft** omdat het nog steeds op iets kan duiden => het ontbreken van een groot genoegen category maar is niet onbekend.

Interresant om te zien is dat de verdeling van merken **niet even ligt** en sommige veel meer voorkomen.

Het resultaat is **21 gedropte kollomen**.

### vehicle_age

In [ ]:
vehicle_age = df['vehicle_age_years']

print("Kollom snel overzicht: \n")
print(f"Kollom null waardes: {vehicle_age.isnull().sum()} ({(vehicle_age.isnull().sum()/totaal_records * 100).round(2)}%)")
print(f"Kollom gemiddelde: {vehicle_age.mean()}")
print(f"Kollom median: {vehicle_age.median()}")
print(f"Kollom uitschieters: {len(df[(df['vehicle_age_years'] >df['vehicle_age_years'].mean() + df['vehicle_age_years'].std()*2) | (df['vehicle_age_years'] < df['vehicle_age_years'].mean() - df['vehicle_age_years'].std()*2)])}")

sns.boxplot(data=df, y='vehicle_age_years')

df.fillna({'vehicle_age_years':vehicle_age.mean()},inplace=True) # use dictonairy to specify which collum to change
df['vehicle_age_years'] = df['vehicle_age_years'].round() # round because all other calues are whole numbers

print("\nAlgemene Data Na aanpassing")
print(df['vehicle_age_years'].describe())
df


Vehicle_age_years is een **numerieke kollom**.Het is ook relatief **normaal verdeelt**. Dit betekend dat we niet extra waarden moeten weg halen voor uitschieters en dat we veilig de null waarde kunnen **imputeren**. Deze imputatie wordt gedaan met **ronding** omdat alle waarde gehele getallen zijn.

### mileage_km

In [ ]:
df = df[
    (df['mileage_km'] <= 500_000) &   # verwijder >500k
    (df['mileage_km'] >= 0) &        # verwijder negatieve waarden
    (df['mileage_km'].notna())       # verwijder NaN
]

Mileage_km is een **numerieke kollom**. Wanneer je eerst de data bekijkt zie je dat deze kollom een harde right skew heeft. Dit heeft als gevolg dat het gemiddelde een slecht maat is om te gebruiken. De median is hier beter met **120361.02** wat dicht bij ligt bij de waarde van een auto van 10 jaar volgens een snelle zoekopdracht. 

Door de heel extreme skewing in de data gaat zelfs **2 * standaard divatie niet echt betrouwbaar** meer zijn dus ik heb geopteerd om the **95% percentiel** te gebruiken. Alle waarde boven dit worden vervangen door een estimatie van wat het zou moeten zijn **12036 * auto leeftijd** want het gemiddelde is niet betrouwbaar. Het zelfde wordt gedaan met negatieve waarde want deze zouden niet mogelijk zijn en null waarde. 

Hierdoor kunnen we nog steeds zoveel mogelijk data behouden. Toch moeten er records gedropped worden voor niet logische zoals een auto die net nieuw is maar veel heeft gereden dit coste **50 records**.

### engine_hours

In [ ]:
engine_hours = df['engine_hours']

print("Kollom snel overzicht: \n")
print(f"Kollom null waardes: {engine_hours.isnull().sum()} ({(engine_hours.isnull().sum()/totaal_records * 100).round(2)}%)")
print(f"Kollom gemiddelde: {engine_hours.mean()}")
print(f"Kollom median: {engine_hours.median()}")
print(f"Kollom uitschieters: {len(df[(df['engine_hours'] >df['engine_hours'].mean() + df['engine_hours'].std()*2) | (df['engine_hours'] < df['engine_hours'].mean() - df['engine_hours'].std()*2)])}")

df.loc[df['engine_hours'] < 0, 'engine_hours'] = df.loc[df['engine_hours'] >= 0, 'engine_hours'].median() # replace negative values with mean without negatives
print(f"Kollom gemiddelde: {df['engine_hours'].mean()}")
print(f"Kollom median: {df['engine_hours'].median()}")
sns.boxplot(data=df, y='engine_hours')

df.fillna({'engine_hours':engine_hours.median()},inplace=True) # use dictonairy to specify which collum to change
df = df.fillna({'engine_hours': df['engine_hours'].median()})
df = df.drop(df[(df['engine_hours'] > 9000) & (df['vehicle_age_years'] < 3)].index) # drop not logical values => having a age of 1 with 8500 + engine hours (.index gives the index of where the condition is true)

print("\nAlgemene Data Na aanpassing")
print(df['engine_hours'].describe())
df

Engine_hours is een **numerieke kollom**. Deze kollom is **vergelijkbaar met de milleage_km kollom** maar zonder de skew. Het meest op opvallend is de **11 negatieve waarde** in de kollom wat normaal gezien niet mogelijk zou moeten zijn. Deze worden dan naar het **gemiddelden gezet** (zonder hun meegerekend) samen met de null waardes.

Deze kollom heeft een paar records die ook onlogisch kunnen zijn. Dit koste **4 records**

### last_service_km_ago

In [ ]:
last_service_km_ago = df['last_service_km_ago']

print("Kollom snel overzicht: \n")
print(f"Kollom null waardes: {last_service_km_ago.isnull().sum()} ({(last_service_km_ago.isnull().sum()/totaal_records * 100).round(2)}%)")
print(f"Kollom gemiddelde: {last_service_km_ago.mean()}")
print(f"Kollom median: {last_service_km_ago.median()}")
print(f"Kollom uitschieters: {len(df[(df['last_service_km_ago'] >df['last_service_km_ago'].mean() + df['last_service_km_ago'].std()*2) | (df['last_service_km_ago'] < df['last_service_km_ago'].mean() - df['last_service_km_ago'].std()*2)])}")

sns.boxplot(data=df, y='last_service_km_ago')

df.fillna({'last_service_km_ago': last_service_km_ago.mean()},inplace=True) # use dictonairy to specify which collum to change

df = df.drop(df[df['last_service_km_ago'] > df['mileage_km']].index) # drop not logical values => last service cannot be higher then the mileage of the car because to have distance between services the car also gets milleage

print("\nAlgemene Data Na aanpassing")
print(df['last_service_km_ago'].describe())
df

Last_service_km_ago is een **numerieke kollom**. Het is een redelijk simpele kollom het is right skewed maar is nog steeds logisch **40592.195120km** rijden is nog steeds mogelijk, enkel onwaarschijnlijk. Via een snelle zoekopdracht vindt je dat **15 000km het gemiddelde is tussen service controles** wat opvallend hoger ligt dan dit gemiddelde van de kollom. Dit komt vast omdat de kollom aanduid wanneer de test voor crash gebeurt is waardoor het logisch is dat het gemiddelde lager ligt.

Er worden well nog wat records gedropt. Als de milleage_km kleiner is dan de last_service_km_ago dan is er iets fout. Dit koste **18 records**.

### oil_quality_pct

In [ ]:
oil_quality_pct = df['oil_quality_pct']

print("Kollom snel overzicht: \n")
print(f"Kollom null waardes: {oil_quality_pct.isnull().sum()} ({(oil_quality_pct.isnull().sum()/totaal_records * 100).round(2)}%)")
print(f"Kollom gemiddelde: {oil_quality_pct.mean()}")
print(f"Kollom median: {oil_quality_pct.median()}")
print(f"Kollom uitschieters: {len(df[(df['oil_quality_pct'] >df['oil_quality_pct'].mean() + df['oil_quality_pct'].std()*2) | (df['oil_quality_pct'] < df['oil_quality_pct'].mean() - df['oil_quality_pct'].std()*2)])}")

sns.boxplot(data=df, y='oil_quality_pct')

df.fillna({'oil_quality_pct': oil_quality_pct.mean()},inplace=True) # use dictonairy to specify which collum to change

df.loc[df['oil_quality_pct'] > 100, 'oil_quality_pct'] = df['oil_quality_pct'].mean() # this collum is a percentage and cannot be higherthen 100, replace it with mean

print("\nAlgemene Data Na aanpassing")
print(df['oil_quality_pct'].describe())
df

Oil_quality_pct is een **numerieke kollom en een percentage**. Het is **heel normaal verdeelt** en zijn er weinig uitschieters. Hier wordt ook weer zoals alteid de null waarde door het gemiddelde vervangen.

Voor logische percentages te hebben werden de **percentages goter naar honderd naar het gemiddelde** gezet.

### avg_trip_length_km

In [ ]:
avg_trip_length_km = df['avg_trip_length_km']

print("Kollom snel overzicht: \n")
print(f"Kollom null waardes: {avg_trip_length_km.isnull().sum()} ({(avg_trip_length_km.isnull().sum()/totaal_records * 100).round(2)}%)")
print(f"Kollom gemiddelde: {avg_trip_length_km.mean()}")
print(f"Kollom median: {avg_trip_length_km.median()}")
print(f"Kollom uitschieters: {len(df[(df['avg_trip_length_km'] >df['avg_trip_length_km'].mean() + df['avg_trip_length_km'].std()*2) | (df['avg_trip_length_km'] < df['avg_trip_length_km'].mean() - df['avg_trip_length_km'].std()*2)])}")

sns.boxplot(data=df, y='avg_trip_length_km')

df.fillna({'avg_trip_length_km': avg_trip_length_km.mean()},inplace=True) # use dictonairy to specify which collum to change

df.loc[df['avg_trip_length_km'] < 0, 'avg_trip_length_km'] = df['avg_trip_length_km'].median() # this collum is a distance and cannot be negative, replace it with mean

print("\nAlgemene Data Na aanpassing")
print(df['avg_trip_length_km'].describe())
df

Avg_trip_length_km is een **numerieke kollom**. Het is **redelijk rechtsgeskewed** te zien aan de boxplot. De waarde die er aanwezig zijn, zijn redelijk reël waardoor **niets aan de data moet aangepast worden** behalve de **null waarde vervangen met de median** want gemiddelde is minder betrouwbaar.

### weather_exposure

In [ ]:
weather_exposure= df['weather_exposure']
print(f"Kollom null waardes: {weather_exposure.isnull().sum()} ({(weather_exposure.isnull().sum()/totaal_records * 100).round(2)}%)")

print(df.weather_exposure.value_counts())
df['weather_exposure'].value_counts().plot(kind='bar',ylabel='Aantal voorkomen')


df.dropna(subset='weather_exposure', inplace=True) # subset is gebruikt om enkel 1 kollom de records te droppen
df

Weather_exposure is een **categorische kollom**. Dit volgt dezelfde logica als bij vehicle_brand waar we the null waarde droppen **ten koste van 12 records** en behouden de other waarden om niet de grootste waarde over te representeren.

In de data set komt medium weer verstoring het meest voor.

### fuel_type

In [ ]:
fuel_type= df['fuel_type']
print(f"Kollom null waardes: {fuel_type.isnull().sum()} ({(fuel_type.isnull().sum()/totaal_records * 100).round(2)}%)")

print(df.fuel_type.value_counts())
df['fuel_type'].value_counts().plot(kind='bar',ylabel='Aantal voorkomen')


df.dropna(subset='fuel_type', inplace=True) # subset is gebruikt om enkel 1 kollom de records te droppen
df

Fuel_type is een **catigorische kollom**. Net zoals de andere worden de **null waarden gedropt met 14 records** in totaal. De other kollom blijft ook weer behouden om niet een andere kollom te iver representeren.

De grootste aantal auto's gebruiken in de dataset fosiele brandstof.

### cleanliness_score

In [ ]:
cleanliness_score = df['cleanliness_score']

print("Kollom snel overzicht: \n")
print(f"Kollom null waardes: {cleanliness_score.isnull().sum()} ({(cleanliness_score.isnull().sum()/totaal_records * 100).round(2)}%)")
print(f"Kollom gemiddelde: {cleanliness_score.mean()}")
print(f"Kollom median: {cleanliness_score.median()}")
print(f"Kollom uitschieters: {len(df[(df['cleanliness_score'] >df['cleanliness_score'].mean() + df['cleanliness_score'].std()*2) | (df['cleanliness_score'] < df['cleanliness_score'].mean() - df['cleanliness_score'].std()*2)])}")

df.loc[df['cleanliness_score'] < 0,'cleanliness_score'] = df.loc[df['cleanliness_score'] > 0,'cleanliness_score'].mean() # replace the 1 negative value with the mean
df.loc[df['cleanliness_score'] > 100,'cleanliness_score'] = df.loc[df['cleanliness_score'] < 100,'cleanliness_score'].mean() # cleanliness_score has a range to 100 and cannot be higher so replace those with the mean
sns.boxplot(data=df, y='cleanliness_score')


df.fillna({'cleanliness_score': cleanliness_score.mean()},inplace=True) # use dictonairy to specify which collum to change


print("\nAlgemene Data Na aanpassing")
print(df['cleanliness_score'].describe())
df

Cleanliness_score is een **numerique kollom**.  Het is **normaal verdeelt** en opvallend **veel null waarde**. Hier kunnen deze waarde weer door het gemiddelde vervangen worden.

De cleanliness_score is een **waarde tussen 0 en 100**. Andere waarde moeten vervangen/gedropt worden. Net zoals alteid worden ze **vervangen** door het gemiddelde om de data te behouden.

### driver_satisfaction_score

In [ ]:
driver_satifaction_score = df['driver_satisfaction_score']

print("Kollom snel overzicht: \n")
print(f"Kollom null waardes: {driver_satifaction_score.isnull().sum()} ({(driver_satifaction_score.isnull().sum()/totaal_records * 100).round(2)}%)")
print(f"Kollom gemiddelde: {driver_satifaction_score.mean()}")
print(f"Kollom median: {driver_satifaction_score.median()}")
print(f"Kollom uitschieters: {len(df[(df['driver_satisfaction_score'] >df['driver_satisfaction_score'].mean() + df['driver_satisfaction_score'].std()*2) | (df['driver_satisfaction_score'] < df['driver_satisfaction_score'].mean() - df['driver_satisfaction_score'].std()*2)])}")

df.loc[df['driver_satisfaction_score'] > 10,'driver_satisfaction_score'] = df.loc[df['driver_satisfaction_score'] < 10,'driver_satisfaction_score'].mean() # driver_satisfaction_score has a range from 1 to 10 and cannot be higher so replace those with the mean
sns.boxplot(data=df, y='driver_satisfaction_score')


df.fillna({'driver_satisfaction_score': driver_satifaction_score.mean()},inplace=True) # use dictonairy to specify which collum to change


print("\nAlgemene Data Na aanpassing")
print(df['driver_satisfaction_score'].describe())
df

Driver_satisfaction_score is een **numerieke kollom**. Het is **redelijk normaal verdeelt** met enkele uitzonderingen die redelijk normaal zijn bij een score zoals dit. De null waarde worden zoals alteid door het **gemiddelde vervangen**.

De score heeft een **range van 0-10** enkele waarden zijn hoger en moeten ook vervangen worden

### tyre_type

In [ ]:
tyre_type= df['tyre_type']
print(f"Kollom null waardes: {tyre_type.isnull().sum()} ({(tyre_type.isnull().sum()/totaal_records * 100).round(2)}%)")

print(df.tyre_type.value_counts())
df['tyre_type'].value_counts().plot(kind='bar',ylabel='Aantal voorkomen')


df.dropna(subset='tyre_type', inplace=True) # subset is gebruikt om enkel 1 kollom de records te droppen
df

Tyre_type is een **categorische kollom**. De kollomen (behalve other) zijn **redelijk even verdeelt**. Net zoals alteid bij categorische data droppen we de null waardes deze keer met **14 records** en behouden we de other waarde om niet te veel andere categorieën te overrepresenteren.

### Conclusie data scrubbing

Uit de bovenstaande scrubbing poging van de data kunnen een aantal dingen zien.
1. Bij categorische data worden alteid de null waarde gedropt. Deze zijn moeilijk bij andere te plaatsen zonder dat een groep meer voorkomt.
2. Bij categorische data wordt alteid de other groep behouden. Ookal is het geen concreet gegeven vervangen zou het zelfde probleem geven als bij null waardes en droppen zou de all kleine data set nog meer verkleinen.
3. Bij numerieke data worden null waarde nooit gedropt maar vervangen met de meeste passende oplossing. 
4. Bij numerieke data wordt enkel data gedropt als het niet logisch is + lastig ergens anders te plaatsen is engine_hours die gerelateerd is aan vehicle_age_years.

Dit allemaal is een heel conservatieve aanpak om zoveel mogelijkdata te bewaren als kan. Het heeft als voordeel om de **data te behouden (nog 917)** voor andere kollomen maar heeft als nadeel dat het de data **meer algemeen maakt en dus belangrijke uitzonderingen kan missen**. Ik denk dat dit het waard is doordat de dataset redelijk klein is en het bewaren van de grote erg belangrijk is voor beter resultaten  zelfs als het meer algemeen is.

## Data analyse + hypotheses

Ok, na het het scrubben van de data gaan we nog kort de data annalyseren via een correlatie matrix en hier hypotheses uit maken.

In [ ]:
# df.loc[df['mileage_km'] < 400_000, 'mileage_km'] = (
#     df.loc[df['mileage_km'] < 400_000, 'vehicle_age_years']
#     * df['mileage_km'].median()
# )



from pandas.plotting import scatter_matrix
scatter_matrix(df[['vehicle_age_years','mileage_km','engine_hours','last_service_km_ago','oil_quality_pct','avg_trip_length_km','cleanliness_score','driver_satisfaction_score']], alpha=0.2, figsize=(10,10))
df[['vehicle_age_years','mileage_km','engine_hours','last_service_km_ago','oil_quality_pct','avg_trip_length_km','cleanliness_score','driver_satisfaction_score']].corr()

In [ ]:
plt.figure(figsize=(6,6))

df.boxplot(column="last_service_km_ago", by="breakdown_next_30_days")

plt.xlabel("Breakdown Next 30 Days")
plt.ylabel("Last Service KM Ago")
plt.title("Service KM Distribution by Breakdown")
plt.suptitle("")
plt.show()

wij halen voor elke data de outliers wge die niet in de normen van de boxplot komn. dit doen we omdat zo als hierboven bij last service zijn er veel die hun auto niet naar de garage brengen maar toch geen breakdown krijgen. door deze outliers weg te halen gaat er rapper de neiging gebracht worden om 1 te gebruiken.een negatiev gevolg is door mindere 0 gaat de neiging van kapot ook bij niet kapot geraakt zijn


df = df[~(
    (df["breakdown_next_30_days"] == 0) &
    (df["last_service_km_ago"] > 30000)
)]
df = df[~(
    (df["breakdown_next_30_days"] == 0) &
    (df["engine_hours"] > 9000)
)]

df = df[~(
    (df["breakdown_next_30_days"] == 0) &
    (df["avg_trip_length_km"] > 100)
)]
df

## Export to file

In [ ]:
df = df[~(
    
    df["last_service_km_ago"] > 30000
)]
df = df[~
    (
    df["engine_hours"] > 9000
)]

 
df = df[~(
    
    df["avg_trip_length_km"] > 100
)]

scatter_matrix(df[['vehicle_age_years','mileage_km','engine_hours','last_service_km_ago','oil_quality_pct','avg_trip_length_km','cleanliness_score','driver_satisfaction_score']], alpha=0.2, figsize=(10,10))
df[['vehicle_age_years','mileage_km','engine_hours','last_service_km_ago','oil_quality_pct','avg_trip_length_km','cleanliness_score','driver_satisfaction_score']].corr()

## Linear regression

In [ ]:
from sklearn.linear_model import LinearRegression
X = df['engine_hours'].values.reshape(-1,1)
y = df['mileage_km'].values

model = LinearRegression().fit(X,y)
model.score(X,y)
y_pred = model.predict(X)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(x=X,y=y)
plt.xlabel('Engine Hours')
plt.ylabel('Mileage (km)')
plt.plot(X,y_pred, color='black')
plt.plot(X,y_pred + 28000, color='green')
plt.plot(X,y_pred - 28000, color='green')

In [ ]:
plt.scatter(x=df['vehicle_age_years'],y=y, alpha=0.5)

In [ ]:

# outliers = []

# for i in range(df.shape[0]):
#     if df.iloc[i]['mileage_km'] > y_pred[i] + 28000 or df.iloc[i]['mileage_km'] < y_pred[i] - 28000:
#         outliers.append(i)

# df_breakdown = df.iloc[outliers]  # dataframe with only outliers
# df = df.drop(index=outliers)      # dataframe without outliersliers]
# df = df.drop(outliers)


In [ ]:
df["km_per_engine_hour"] = df["mileage_km"] / (df["engine_hours"] + 1)
df["service_ratio"] = df["last_service_km_ago"] / (df["mileage_km"] + 1)

In [ ]:
df.to_csv('../../csv/new_train_data/train_CarBreakDown.csv')